# 003 — Evaluation Metrics + First Baseline (GlobalSTD)



# **1. IMPORTS**

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
try:
    import portion as P
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "portion"])
    import portion as P
print(f"pandas version: {pd.__version__}")

pandas version: 2.2.2


# **2. MOUNT GOOGLE DRIVE (THE PROJECT IS PREFERRED TO BE RUN ON GOOGLE COLAB)**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
IN_COLAB = True
DRIVE_ROOT = /content/drive/MyDrive/BeaconProject


# **3. CONFIGURATION**

This notebook needs two things per mission: the **preprocessed** train/test
CSVs from notebook `002_ESA_PREPROCESSING.ipynb`, and the **raw** `labels.csv`/`anomaly_types.csv` —
we score against the original per-event annotations (with their real event
IDs), not a re-derived version, so that multi-fragment events are still
treated as one event (Appendix C.2.4) rather than accidentally split apart.


In [ ]:
MISSION_CONFIG = {
    "ESA-Mission1": {
        "lightweight_channels": [f"channel_{i}" for i in range(41, 47)],
        "test_data_split": "2007-01-01",
        "resampling_rule": pd.Timedelta(seconds=30),  # matches notebook 002_ESA_PREPROCESSING.ipynb exactly
        # paper Table 4, lightweight subset, "all events" (Communication Gap excluded, Table 16 caption)
        "table4_globalstd": {
            "GlobalSTD3": {"precision": 0.001, "recall": 0.431, "f0.5": 0.001},
            "GlobalSTD5": {"precision": 0.288, "recall": 0.169, "f0.5": 0.253},
        },
    },
    "ESA-Mission2": {
        "lightweight_channels": [f"channel_{i}" for i in range(18, 29)],
        "test_data_split": "2001-10-01",
        "resampling_rule": pd.Timedelta(seconds=18),  # matches notebook 002_ESA_PREPROCESSING.ipynb exactly
        "table4_globalstd": {
            "GlobalSTD3": {"precision": 0.006, "recall": 1.000, "f0.5": 0.007},
            "GlobalSTD5": {"precision": 0.061, "recall": 1.000, "f0.5": 0.075},
        },
    },
}

# --- EDIT THIS ONE LINE to switch missions ---
ACTIVE_MISSION = "ESA-Mission1"

CFG = MISSION_CONFIG[ACTIVE_MISSION]
TARGET_CHANNELS = CFG["lightweight_channels"]
RESAMPLING_RULE = CFG["resampling_rule"]

RAW_DATA_ROOT = DRIVE_ROOT / ACTIVE_MISSION / ACTIVE_MISSION
PREPROCESSED_DIR = DRIVE_ROOT / "preprocessed" / ACTIVE_MISSION  # notebook 002_ESA_PREPROCESSING.ipynb output

BETA = 0.5  # paper's primary metric weighting -- penalizes false alarms harder than misses

print(f"Active mission: {ACTIVE_MISSION}")
print(f"Target channels: {TARGET_CHANNELS}")
print(f"Reading preprocessed data from: {PREPROCESSED_DIR}")
print(f"Reading raw labels from: {RAW_DATA_ROOT}")


Active mission: ESA-Mission1
Target channels: ['channel_41', 'channel_42', 'channel_43', 'channel_44', 'channel_45', 'channel_46']
Reading preprocessed data from: /content/drive/MyDrive/BeaconProject/preprocessed/ESA-Mission1
Reading raw labels from: /content/drive/MyDrive/BeaconProject/ESA-Mission1/ESA-Mission1


# **4. LOAD PREPROCESSED TRAIN/TEST(notebook 002_ESA_PREPROCESSING's OUTPUT)**

In [ ]:
anomaly_cols = [f"is_anomaly_{ch}" for ch in TARGET_CHANNELS]
dtypes = {ch: np.float32 for ch in TARGET_CHANNELS}
dtypes.update({c: np.uint8 for c in anomaly_cols})

train_df = pd.read_csv(PREPROCESSED_DIR / "train.csv", index_col="timestamp", parse_dates=True, dtype=dtypes)
test_df = pd.read_csv(PREPROCESSED_DIR / "test.csv", index_col="timestamp", parse_dates=True, dtype=dtypes)

print(f"train: {train_df.shape}, test: {test_df.shape}")
print(f"train range: {train_df.index.min()} -> {train_df.index.max()}")
print(f"test range:  {test_df.index.min()} -> {test_df.index.max()}")


train: (7099200, 12), test: (7364161, 12)
train range: 2000-01-01 00:00:00 -> 2006-09-30 23:59:30
test range:  2007-01-01 00:00:00 -> 2014-01-01 00:00:00


# **5. LOAD RAW EVENT ANNOTATIONS**

Same tz-normalization as notebook 002 (real ESA pickles are commonly tz-aware;
`labels.csv` parses tz-naive by default).

**Important correction, found by checking the reference orchestration script
(`mission1_experiments.py`) directly:** Table 4's numbers exclude Communication
Gap events from *detection scoring* (Table 16 caption), but the reference
implementation does this via `select_labels={"Category": ["Rare Event",
"Anomaly"]}` passed to the metric at scoring time — **not** by deleting gap
rows from the data beforehand. That distinction matters: the metric's
false-positive correction subtracts *every* annotated event (gaps included)
from what counts as "nominal" time, specifically so a detector firing during a
genuine communication gap (where there's no reliable signal at all) isn't
punished as if it raised a false alarm during ordinary quiet operation. Drop
the gap rows entirely beforehand, and that time gets miscounted as nominal —
which is exactly what this notebook was doing until now. Fixed below: keep all
rows, attach `Category`, and let the scorer's `select_labels` do the filtering
at the right stage.


In [ ]:
labels_df = pd.read_csv(RAW_DATA_ROOT / "labels.csv", parse_dates=["StartTime", "EndTime"])
anomaly_types_df = pd.read_csv(RAW_DATA_ROOT / "anomaly_types.csv")

if labels_df["StartTime"].dt.tz is not None:
    labels_df["StartTime"] = labels_df["StartTime"].dt.tz_localize(None)
if labels_df["EndTime"].dt.tz is not None:
    labels_df["EndTime"] = labels_df["EndTime"].dt.tz_localize(None)

# attach Category (needed for select_labels) -- keep ALL rows, including gaps
labels_df = labels_df.merge(anomaly_types_df[["ID", "Category"]], on="ID", how="left")

print(f"Total event rows (all categories, all channels): {len(labels_df)}")
print(f"Category breakdown: {anomaly_types_df['Category'].value_counts().to_dict()}")


Total event rows (all categories, all channels): 3589
Category breakdown: {'Anomaly': 118, 'Rare Event': 78, 'Communication Gap': 4}


# **6. THE METRIC: CORRECTED EVENT-WISE F-BETA**

Extracted and adapted from the reference implementation
(`timeeval/metrics/ESA_ADB_metrics.py`), scoped down to just the event-wise
score + alarming precision (channel-aware, ADTQC, and affiliation-based scores
are deferred).

This is the paper's **highest-priority metric** (Table 3, group 1): properly
identifying events while strongly penalizing false alarms. The key idea beyond
plain event-wise precision (Hundman et al., 2018): a naive version would let a
detector that flags *every single sample* score a perfect precision, since it
technically overlaps every real event too. The Sehili & Zhang (2023) correction
fixes this by also penalizing the *fraction of nominal time* that got falsely
alarmed:

`Precision = (TP_events / (TP_events + FP_events)) × (1 − FP_seconds / Nominal_seconds)`


In [ ]:
def convert_time_series_to_events(vector) -> P.Interval:
    '''Turn a (timestamp, binary) series into closed/closed-open intervals
    representing contiguous runs of 1s.'''
    vector = np.asarray(vector)

    def find_runs(x):
        x = np.asanyarray(x)
        n = x.shape[0]
        if n == 0:
            return np.array([]), np.array([]), np.array([])
        loc_run_start = np.empty(n, dtype=bool)
        loc_run_start[0] = True
        np.not_equal(x[:-1], x[1:], out=loc_run_start[1:])
        run_starts = np.nonzero(loc_run_start)[0]
        run_values = x[loc_run_start]
        run_lengths = np.diff(np.append(run_starts, n))
        run_ends = run_starts + run_lengths
        return np.stack((run_starts[run_values > 0], run_ends[run_values > 0])).transpose()

    non_zero_runs = find_runs(vector[..., 1])
    events = []
    n = len(vector)
    for x, y in non_zero_runs:
        if y == n:
            events.append(P.closed(vector[..., 0][x], vector[..., 0][y - 1]))
        else:
            events.append(P.closedopen(vector[..., 0][x], vector[..., 0][y]))
    return P.Interval(*events)


class EventWiseScorer:
    '''Corrected event-wise precision/recall/F-beta + alarming precision.'''

    NANOSECONDS_IN_SECOND = 1e9

    def __init__(self, betas=1.0, select_labels=None, full_range=None):
        self._betas = np.atleast_1d(betas)
        self.full_range = full_range
        if select_labels is None or len(select_labels) == 0:
            self.selected_labels = {}
        else:
            self.selected_labels = {c: np.atleast_1d(v) for c, v in select_labels.items()}

    def score(self, y_true: pd.DataFrame, y_pred) -> dict:
        '''
        y_true: DataFrame with ID, StartTime, EndTime (+ any columns referenced
                in select_labels, e.g. Category). One row per event, or per
                event-fragment sharing the same ID -- those get unioned.
        y_pred: list/array of (timestamp, is_anomaly) pairs, is_anomaly in {0,1}.

        NOTE: select_labels only restricts which events count toward TP/FN.
        The false-positive/nominal-time correction always uses the FULL,
        unfiltered y_true -- e.g. a Communication Gap excluded via
        select_labels still needs to be subtracted from "nominal" time, since
        a detection firing during a real gap (no reliable signal) shouldn't be
        punished as a false alarm during ordinary quiet operation. This
        matches the reference implementation exactly (verified against it).
        '''
        y_pred = np.asarray(y_pred)

        if self.full_range is None:
            self.full_range = (min(y_true["StartTime"].min(), min(y_pred[..., 0])),
                               max(y_true["EndTime"].max(), max(y_pred[..., 0])))
        if y_pred[0, 0] > self.full_range[0]:
            y_pred = np.array([np.array([self.full_range[0], y_pred[0, 1]]), *y_pred])
        if y_pred[-1, 0] < self.full_range[1]:
            y_pred = np.array([*y_pred, np.array([self.full_range[1], y_pred[-1, 1]])])

        events_pred = convert_time_series_to_events(y_pred)

        filtered_y_true = y_true.copy()
        for col, val in self.selected_labels.items():
            filtered_y_true = filtered_y_true[filtered_y_true[col].isin(val)]

        true_positives = 0
        false_negatives = 0
        redundant_detections = 0
        matched_events_pred = [False for _ in events_pred]

        for aid in filtered_y_true["ID"].unique():
            gt = filtered_y_true[filtered_y_true["ID"] == aid]
            gt_intervals = P.Interval(*[P.closed(*row) for _, row in gt[["StartTime", "EndTime"]].iterrows()])

            already_detected = [0 for _ in gt_intervals]
            at_least_one_detected = False
            for p, pred in enumerate(events_pred):
                if pred.upper < gt_intervals.lower or pred.lower > gt_intervals.upper:
                    continue
                intersections = [not (pred & g).empty for g in gt_intervals]
                if not any(intersections):
                    continue
                matched_events_pred[p] = True
                if not at_least_one_detected:
                    true_positives += 1
                    at_least_one_detected = True
                for i, val in enumerate(intersections):
                    if val:
                        already_detected[i] += 1

            for det in already_detected:
                if det > 1:
                    redundant_detections += (det - 1)
            if not at_least_one_detected:
                false_negatives += 1

        # events_gt intentionally uses the FULL y_true (not filtered_y_true) --
        # see the note in this method's docstring.
        events_gt = P.Interval(*[P.closed(*row) for _, row in y_true[["StartTime", "EndTime"]].iterrows()])
        false_positives = sum(
            1 for pred, matched in zip(events_pred, matched_events_pred)
            if not matched and (pred & events_gt).empty
        )

        divider = true_positives + false_positives
        precision = 0.0 if divider == 0 else true_positives / divider

        divider = true_positives + redundant_detections
        alarming_precision = 0.0 if divider == 0 else true_positives / divider

        if precision > 0:
            nominal_interval = P.closed(*self.full_range) - events_gt
            false_positives_interval = nominal_interval & events_pred
            nominal_seconds = sum((iv.upper - iv.lower).value / self.NANOSECONDS_IN_SECOND for iv in nominal_interval)
            fp_seconds = sum((iv.upper - iv.lower).value / self.NANOSECONDS_IN_SECOND for iv in false_positives_interval)
            tnr = 1 - fp_seconds / nominal_seconds if nominal_seconds > 0 else 1.0
            precision *= tnr

        divider = true_positives + false_negatives
        recall = 0.0 if divider == 0 else true_positives / divider

        result = {
            "TP": true_positives, "FP": false_positives, "FN": false_negatives,
            "alarming_precision": alarming_precision,
            "EW_precision": precision, "EW_recall": recall,
        }
        for b in self._betas:
            divider = b ** 2 * precision + recall
            result[f"EW_F_{b:.2f}"] = 0.0 if divider == 0 else ((1 + b ** 2) * precision * recall) / divider
        return result


# **7. PROOF BEFORE REAL NUMBERS**
Three checks, all against known, hand-checkable behavior — not just "it runs
without error":

1. **The authors' own worked example**, `select_labels`-filtered exactly as in
   their own `ESA_ADB_metrics.py` `__main__` block. Ran the original,
   unmodified reference implementation on this exact input in a separate
   environment: `EW_precision=0.7272727272727273, EW_recall=1.0,
   EW_F_0.50=0.7692307692307693, alarming_precision=1.0`. Asserts our
   extraction matches those exactly.
2. **The "always alarming" scenario** the correction exists to prevent
   (paper's Figure 12, "Algorithm 2"): a detector that flags every single
   sample gets perfect *recall* but should be crushed to **zero precision**,
   because it never has a quiet nominal moment.
3. **The gap-handling fix from section 3**: a detector that correctly catches
   two real anomalies, plus briefly (and harmlessly) fires during an
   unrelated Communication Gap, should score a perfect 1.0 — not be punished
   for the gap. This is the exact scenario that was silently wrong before.


In [ ]:
# --- check 1: matches the reference implementation's own worked example ---
_full_range = (pd.to_datetime("2015-01-01"), pd.to_datetime("2015-01-15"))
_y_true = pd.DataFrame([
    ["id_0", pd.to_datetime("2015-01-01"), pd.to_datetime("2015-01-02"), "Multivariate", "Point"],
    ["id_1", pd.to_datetime("2015-01-04"), pd.to_datetime("2015-01-05"), "Univariate", "Subsequence"],
    ["id_2", pd.to_datetime("2015-01-07"), pd.to_datetime("2015-01-08"), "Multivariate", "Subsequence"],
], columns=["ID", "StartTime", "EndTime", "Dimensionality", "Length"])
_y_pred = [[pd.to_datetime("2015-01-01"), 0], [pd.to_datetime("2015-01-04"), 1], [pd.to_datetime("2015-01-09"), 0]]

_scorer = EventWiseScorer(betas=0.5, full_range=_full_range,
                          select_labels={"Dimensionality": "Multivariate", "Length": "Subsequence"})
_result = _scorer.score(_y_true, _y_pred)
print("Check 1 result:", _result)

assert abs(_result["EW_precision"] - 0.7272727272727273) < 1e-9
assert abs(_result["EW_recall"] - 1.0) < 1e-9
assert abs(_result["EW_F_0.50"] - 0.7692307692307693) < 1e-9
assert abs(_result["alarming_precision"] - 1.0) < 1e-9
print("PASS: matches the reference implementation's own worked example exactly.\n")

# --- check 2: always-alarming detector is crushed to 0 precision ---
_y_true_plain = _y_true[["ID", "StartTime", "EndTime"]]
_y_pred_always_on = [[pd.to_datetime("2015-01-01"), 1]]

_scorer2 = EventWiseScorer(betas=0.5, full_range=_full_range)
_result2 = _scorer2.score(_y_true_plain, _y_pred_always_on)
print("Check 2 result:", _result2)

assert _result2["EW_recall"] == 1.0, "always-on should trivially detect everything"
assert _result2["EW_precision"] == 0.0, "STOP: the false-positive-duration correction is not penalizing an always-on detector"
print("PASS: an always-alarming detector is correctly crushed to 0 precision.\n")

# --- check 3: firing during a Communication Gap must not be punished ---
_y_true_gap = pd.DataFrame([
    ["id_0", pd.to_datetime("2015-01-01"), pd.to_datetime("2015-01-02"), "Anomaly"],
    ["id_1", pd.to_datetime("2015-01-04"), pd.to_datetime("2015-01-05"), "Communication Gap"],
    ["id_2", pd.to_datetime("2015-01-07"), pd.to_datetime("2015-01-08"), "Anomaly"],
], columns=["ID", "StartTime", "EndTime", "Category"])
_y_pred_gap = [[pd.to_datetime("2015-01-01"), 1], [pd.to_datetime("2015-01-02"), 0],
               [pd.to_datetime("2015-01-04"), 1], [pd.to_datetime("2015-01-05"), 0],
               [pd.to_datetime("2015-01-07"), 1], [pd.to_datetime("2015-01-08"), 0]]

_scorer3 = EventWiseScorer(betas=0.5, full_range=_full_range, select_labels={"Category": ["Anomaly"]})
_result3 = _scorer3.score(_y_true_gap, _y_pred_gap)
print("Check 3 result:", _result3)

assert _result3["EW_precision"] == 1.0, "STOP: firing during a Communication Gap is being wrongly punished as a false alarm"
print("PASS: a detection during a Communication Gap is correctly not treated as a false alarm.")


Check 1 result: {'TP': 1, 'FP': 0, 'FN': 0, 'alarming_precision': 1.0, 'EW_precision': 0.7272727272727273, 'EW_recall': 1.0, 'EW_F_0.50': np.float64(0.7692307692307693)}
PASS: matches the reference implementation's own worked example exactly.

Check 2 result: {'TP': 3, 'FP': 0, 'FN': 0, 'alarming_precision': 1.0, 'EW_precision': 0.0, 'EW_recall': 1.0, 'EW_F_0.50': np.float64(0.0)}
PASS: an always-alarming detector is correctly crushed to 0 precision.

Check 3 result: {'TP': 2, 'FP': 0, 'FN': 0, 'alarming_precision': 1.0, 'EW_precision': 1.0, 'EW_recall': 1.0, 'EW_F_0.50': np.float64(1.0)}
PASS: a detection during a Communication Gap is correctly not treated as a false alarm.


# **8. GLOBALSTD**

The simplest baseline in the paper (`TimeEval-algorithms/std/algorithm.py`,
reference implementation, ~10 lines of real logic): per-channel mean and
standard deviation computed from **nominal-labeled training points only**,
then flag anything beyond `mean ± tol·std`. `tol=3` and `tol=5` are the two
variants the paper reports (Table 15).


In [ ]:
def fit_global_std(train_df: pd.DataFrame, target_channels: list) -> tuple[dict, dict]:
    means, stds = {}, {}
    for ch in target_channels:
        label_col = f"is_anomaly_{ch}"
        nominal_values = train_df.loc[train_df[label_col] == 0, ch]
        means[ch] = nominal_values.mean()
        std = nominal_values.std()
        stds[ch] = std if std > 0 else 1.0  # avoid a zero-width threshold on a constant channel
    return means, stds


def predict_global_std(df: pd.DataFrame, target_channels: list, means: dict, stds: dict, tol: float) -> pd.DataFrame:
    preds = {}
    for ch in target_channels:
        upper = means[ch] + tol * stds[ch]
        lower = means[ch] - tol * stds[ch]
        preds[ch] = ((df[ch] > upper) | (df[ch] < lower)).astype(np.uint8)
    return pd.DataFrame(preds, index=df.index)


train_means, train_stds = fit_global_std(train_df, TARGET_CHANNELS)
print("Per-channel nominal-training mean / std:")
for ch in TARGET_CHANNELS:
    print(f"  {ch}: mean={train_means[ch]:.4f}, std={train_stds[ch]:.4f}")


Per-channel nominal-training mean / std:
  channel_41: mean=0.8112, std=0.0186
  channel_42: mean=0.7848, std=0.0270
  channel_43: mean=0.7724, std=0.0328
  channel_44: mean=0.7973, std=0.0182
  channel_45: mean=0.8134, std=0.0198
  channel_46: mean=0.7687, std=0.0359


# **9. RUN GLOBALSTD3 AND GLOBALSTD5 ON THE REAL TEST SET**

In [ ]:
predictions = {}
for tol, name in [(3.0, "GlobalSTD3"), (5.0, "GlobalSTD5")]:
    predictions[name] = predict_global_std(test_df, TARGET_CHANNELS, train_means, train_stds, tol)
    flagged_pct = predictions[name].values.mean() * 100
    print(f"{name}: flagged {flagged_pct:.4f}% of all (timestamp, channel) pairs in the test set")


GlobalSTD3: flagged 0.7699% of all (timestamp, channel) pairs in the test set
GlobalSTD5: flagged 0.6361% of all (timestamp, channel) pairs in the test set


# **10. SCORE AGAINST REAL EVENTS**
Matching the reference's own usage (`y_true` is the set of events affecting
*any* target channel; `y_pred` is the logical OR across all target channels —
Table 4 reports one number per algorithm per mission, not one per channel).
`select_labels` is applied here, at scoring time, exactly as
`mission1_experiments.py` does it — **not** by pre-filtering `labels_df` (see
section 3's note on why that distinction matters).


In [ ]:
y_true = labels_df[labels_df["Channel"].isin(TARGET_CHANNELS)].copy()
y_true = y_true[y_true["StartTime"] >= pd.to_datetime(CFG["test_data_split"])]

print(f"Test period: {test_df.index.min()} -> {test_df.index.max()}")
print(f"Total events in the test period affecting target channels: {y_true['ID'].nunique()}")
print(f"By category: {y_true.drop_duplicates('ID')['Category'].value_counts().to_dict()}")
print()
print("Full event list for this test period:")
print(y_true[["ID", "Channel", "StartTime", "EndTime", "Category"]]
      .sort_values("StartTime").to_string(index=False))

full_range = (test_df.index.min(), test_df.index.max())

results = {}
for name, pred_df in predictions.items():
    scorer = EventWiseScorer(betas=BETA, full_range=full_range,
                             select_labels={"Category": ["Rare Event", "Anomaly"]})
    combined_pred = pred_df[TARGET_CHANNELS].any(axis=1).astype(np.uint8)
    y_pred_pairs = list(zip(combined_pred.index, combined_pred.values))
    results[name] = scorer.score(y_true, y_pred_pairs)
    print(f"\n{name}: {results[name]}")


Test period: 2007-01-01 00:00:00 -> 2014-01-01 00:00:00
Total events in the test period affecting target channels: 65
By category: {'Rare Event': 36, 'Anomaly': 29}

Full event list for this test period:
    ID    Channel               StartTime                 EndTime   Category
id_116 channel_45 2007-04-14 08:14:16.035 2007-04-20 07:02:25.035    Anomaly
id_116 channel_43 2007-04-14 08:14:16.035 2007-04-20 07:02:25.035    Anomaly
id_116 channel_42 2007-04-14 08:14:16.035 2007-04-20 07:02:25.035    Anomaly
id_116 channel_41 2007-04-14 08:14:16.035 2007-04-20 07:02:25.035    Anomaly
id_116 channel_46 2007-04-14 08:14:16.035 2007-04-20 07:02:25.035    Anomaly
id_116 channel_44 2007-04-14 08:14:16.035 2007-04-20 07:02:25.035    Anomaly
 id_15 channel_46 2007-04-25 15:00:25.035 2007-04-25 15:02:55.035 Rare Event
 id_15 channel_45 2007-04-25 15:00:25.035 2007-04-25 15:02:55.035 Rare Event
 id_15 channel_43 2007-04-25 15:00:25.035 2007-04-25 15:02:55.035 Rare Event
 id_15 channel_42 2007-04-

# **10.1] PER-EVENT BREAKDOWN (EVERY EVENT, NOT JUST THE CURATED HARD ONES)**
Table 7 in the paper deliberately lists the *hardest* events for the
lightweight subset — several are explicitly noted as "much easier to spot in
channels 58-60" or similar, i.e. channels outside 41-46 entirely, so missing
those specific ones is expected, not a bug. This cell instead shows **every**
event in the test period, so it's possible to see the whole picture rather
than just the pre-selected hard cases.


In [ ]:
_buffer = pd.Timedelta(hours=1)  # small tolerance window around each event's boundary

for name, pred_df in predictions.items():
    combined_pred = pred_df[TARGET_CHANNELS].any(axis=1).astype(np.uint8)
    print(f"\n--- {name} ---")
    for eid in y_true["ID"].unique():
        rows = y_true[y_true["ID"] == eid]
        start, end, category = rows["StartTime"].min(), rows["EndTime"].max(), rows["Category"].iloc[0]
        window = combined_pred.loc[start - _buffer: end + _buffer]
        hit = (window == 1).any()
        print(f"  {eid:<18} [{category:<17}] {start} -> {end}: {'DETECTED' if hit else 'missed'}")



--- GlobalSTD3 ---
  id_14              [Rare Event       ] 2013-06-30 19:36:39.687000 -> 2013-07-20 16:49:09.684000: missed
  id_15              [Rare Event       ] 2007-04-25 15:00:25.035000 -> 2007-04-26 00:02:55.035000: missed
  id_18              [Anomaly          ] 2008-09-08 13:46:47.460000 -> 2008-09-08 16:13:24.963000: DETECTED
  id_24              [Rare Event       ] 2012-12-18 06:32:09.723000 -> 2012-12-19 06:01:39.723000: missed
  id_26              [Rare Event       ] 2007-10-09 21:20:55.008000 -> 2007-10-10 00:42:55.008000: missed
  id_27              [Rare Event       ] 2007-10-26 12:26:47.505000 -> 2007-10-26 16:30:25.008000: missed
  id_28              [Rare Event       ] 2007-12-24 02:34:24.999000 -> 2007-12-24 05:16:24.999000: missed
  id_29              [Rare Event       ] 2008-06-07 15:36:47.475000 -> 2008-06-07 16:42:54.975000: missed
  id_31              [Rare Event       ] 2008-06-10 16:20:54.975000 -> 2008-06-10 19:42:54.975000: missed
  id_32              [Ra

# **10.2] BOUNDARY-ARTIFACT CHECK**

A specific, confirmed structural issue, not a guess: NOTEBOOK `002_ESA_PREPROCESSING`'s
point-preserving correction (Appendix C.3, step 3) shifts an anomalous value
*forward* onto the next grid point when it would otherwise be erased by
resampling. GlobalSTD correctly flags that (later) grid point — but the
scorer's event-overlap check (`convert_time_series_to_events`) only builds a
predicted interval *forward* from a detection's timestamp, never backward. So
a detection that correctly landed on the corrected grid point can still miss
the raw event's exact (pre-correction) timestamp, purely from that one-step
shift, even though the detection genuinely caught the event.

Verified directly: constructed a detection at a grid point 20 seconds after a
point event's true raw timestamp (the same bin) — the scorer's strict overlap
check returns `False` even though the detection is, in every practical sense,
a catch.

This cell checks, for every event this run's scorer called a miss, whether a
detection actually fired within one resampling step of the raw boundary. If
so, that's very likely this exact artifact — not a real miss.


In [ ]:
for name, pred_df in predictions.items():
    combined_pred = pred_df[TARGET_CHANNELS].any(axis=1).astype(np.uint8)
    print(f"\n--- {name}: events missed by strict scoring, checked for a nearby detection ---")
    n_boundary_artifacts = 0
    for eid in y_true["ID"].unique():
        rows = y_true[y_true["ID"] == eid]
        start, end = rows["StartTime"].min(), rows["EndTime"].max()

        strict_hit = (combined_pred.loc[start:end] == 1).any()
        if strict_hit:
            continue

        padded_hit = (combined_pred.loc[start - RESAMPLING_RULE: end + RESAMPLING_RULE] == 1).any()
        if padded_hit:
            n_boundary_artifacts += 1
            print(f"  {eid:<18} strict MISS, but detected within one grid step "
                  f"({RESAMPLING_RULE}) -- likely the forward-shift artifact, not a real miss")
        else:
            print(f"  {eid:<18} strict MISS, nothing detected nearby either -- looks like a genuine miss")
    print(f"\n{name}: {n_boundary_artifacts} of the strict misses look like boundary artifacts.")



--- GlobalSTD3: events missed by strict scoring, checked for a nearby detection ---
  id_14              strict MISS, nothing detected nearby either -- looks like a genuine miss
  id_15              strict MISS, nothing detected nearby either -- looks like a genuine miss
  id_24              strict MISS, nothing detected nearby either -- looks like a genuine miss
  id_26              strict MISS, nothing detected nearby either -- looks like a genuine miss
  id_27              strict MISS, nothing detected nearby either -- looks like a genuine miss
  id_28              strict MISS, nothing detected nearby either -- looks like a genuine miss
  id_29              strict MISS, nothing detected nearby either -- looks like a genuine miss
  id_31              strict MISS, nothing detected nearby either -- looks like a genuine miss
  id_32              strict MISS, nothing detected nearby either -- looks like a genuine miss
  id_33              strict MISS, nothing detected nearby either -- l

# **11. A SECOND SCORING PASS, CORRECTING FOR THE KNOWN FORWARD-SHIFT**
Same events, same detections, same metric — the only change is padding each
ground-truth event's boundary by one resampling step (`RESAMPLING_RULE`) on
each side before scoring, to account for the documented, one-directional
shift our own point-preserving correction can introduce. This is **not**
presented as a replacement for the strict, paper-comparable score above — it's
a second, clearly-separate lens, specifically to see how much of the gap is
this boundary effect versus genuinely-missed events.


In [ ]:
results_tolerant = {}
for name, pred_df in predictions.items():
    y_true_padded = y_true.copy()
    y_true_padded["StartTime"] = y_true_padded["StartTime"] - RESAMPLING_RULE
    y_true_padded["EndTime"] = y_true_padded["EndTime"] + RESAMPLING_RULE

    scorer = EventWiseScorer(betas=BETA, full_range=full_range,
                             select_labels={"Category": ["Rare Event", "Anomaly"]})
    combined_pred = pred_df[TARGET_CHANNELS].any(axis=1).astype(np.uint8)
    y_pred_pairs = list(zip(combined_pred.index, combined_pred.values))
    results_tolerant[name] = scorer.score(y_true_padded, y_pred_pairs)
    print(f"{name} (tolerant): {results_tolerant[name]}")


GlobalSTD3 (tolerant): {'TP': 26, 'FP': 315, 'FN': 39, 'alarming_precision': 0.07008086253369272, 'EW_precision': 0.07600128060549187, 'EW_recall': 0.4, 'EW_F_0.50': np.float64(0.0906935829763027)}
GlobalSTD5 (tolerant): {'TP': 24, 'FP': 0, 'FN': 41, 'alarming_precision': 0.6153846153846154, 'EW_precision': 0.9999988751168087, 'EW_recall': 0.36923076923076925, 'EW_F_0.50': np.float64(0.7453411149778075)}


# **12. COMPARE AGAINST THE PAPER'S TABLE 4**
This is the checkpoint that matters: if these line up (even roughly — some
drift is expected since this is the lightweight 6-channel subset and exact
annotation edge cases can differ slightly), both the preprocessing pipeline
from notebook 002_ESA_PREPROCESSING and the scoring logic from this notebook are validated
against ground truth, independent of any complex model.

**Worth calibrating expectations before reading too much into any single
number here.** GlobalSTD is, in the paper's own words, one of the weakest
baselines — "unsupervised algorithms perform very poorly for Mission1... just
slightly better [than random], which is especially disappointing." It's also
the most threshold-sensitive method in the whole benchmark: its output is a
direct, unsmoothed function of exactly two numbers (per-channel mean and std),
so small differences anywhere upstream (floating-point rounding through a
different-but-equally-valid resampling implementation, slightly different
event-boundary handling, channel propagation-time differences between how a
given event's start/end got recorded per-channel) can shift *which specific
events* get caught without anything being wrong. Recall in particular is
computed over a small number of discrete events, not millions of independent
samples — catching or missing 2-3 events swings it by double digits.

The realistic bar for GlobalSTD specifically: **the same qualitative story**
(very weak, low precision, particularly poor on the lightweight subset) and
**the same order of magnitude** — not a match to three decimal places. The
tighter, more meaningful validation comes from the *next* notebook's baselines
(PCC/HBOS/iForest/KNN), which don't depend on a single brittle threshold the
way GlobalSTD does.


In [ ]:
print(f"{'Algorithm':<12} {'Metric':<10} {'Strict':>10} {'Tolerant':>10} {'Paper (Table 4)':>16}")
print("-" * 74)
for name in ["GlobalSTD3", "GlobalSTD5"]:
    paper = CFG["table4_globalstd"][name]
    ours = results[name]
    ours_tol = results_tolerant[name]
    rows = [
        ("precision", ours["EW_precision"], ours_tol["EW_precision"], paper["precision"]),
        ("recall", ours["EW_recall"], ours_tol["EW_recall"], paper["recall"]),
        (f"F_{BETA:.2f}", ours[f"EW_F_{BETA:.2f}"], ours_tol[f"EW_F_{BETA:.2f}"], paper["f0.5"]),
    ]
    for metric_name, strict_val, tol_val, paper_val in rows:
        print(f"{name:<12} {metric_name:<10} {strict_val:>10.4f} {tol_val:>10.4f} {paper_val:>16.4f}")
    print()


Algorithm    Metric         Strict   Tolerant  Paper (Table 4)
--------------------------------------------------------------------------
GlobalSTD3   precision      0.0234     0.0760           0.0010
GlobalSTD3   recall         0.1231     0.4000           0.4310
GlobalSTD3   F_0.50         0.0279     0.0907           0.0010

GlobalSTD5   precision      0.2500     1.0000           0.2880
GlobalSTD5   recall         0.0923     0.3692           0.1690
GlobalSTD5   F_0.50         0.1863     0.7453           0.2530

